# 13장. 사전 학습 모델과 미세 조정

레이블이 없는 영화평 11만 개로 언어 모델을 먼저 학습시키고, 그 모델의 층을 가져와 레이블이 적은 분류 과제에 쓴다. 층을 얼리는 방법과 전체를 미세 조정하는 방법을 레이블의 수를 바꿔 가며 비교한다.

**준비:** 11장, 12장과 같은 파일 `ratings_train.txt`(https://github.com/e9t/nsmc)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 시험용 파일 `ratings_test.txt`는 16장에서 쓴다. 끝까지 실행하는 데 30분쯤 걸린다. 13.5절의 마지막 셀은 `RUN_PUBLIC = False`로 두면 실행되지 않는다.

## 13.1 레이블의 수를 줄여 처음부터 학습하기

In [ ]:
import math, time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

torch.set_num_threads(2)
raw = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
raw = raw.reset_index(drop=True)
order = np.random.default_rng(0).permutation(len(raw))        # 11장과 같은 나누기
train, valid = raw.iloc[order[:30000]], raw.iloc[order[-10000:]]
spare = raw.iloc[order[30000:-10000]]                 # 12장까지 쓰지 않은 문장
texts_tr, texts_va = train["document"].tolist(), valid["document"].tolist()
texts_un = spare["document"].tolist()                 # 이 문장들의 레이블은 읽지 않는다
y_tr = torch.tensor(train["label"].to_numpy())
y_va = torch.tensor(valid["label"].to_numpy())

PAD, UNKNOWN = 0, 1                                 # 0번은 패딩, 1번은 어휘에 없는 글자
counts = Counter(ch for text in texts_tr for ch in text)
vocab = {ch: k + 2 for k, (ch, n) in enumerate(counts.most_common()) if n >= 5}
V = len(vocab) + 2
START, END = V, V + 1                               # 문장의 시작과 끝을 알리는 토큰

def encode(texts, numbers=None, length=64):
    """<시작> + 글자 번호 + <끝>. length + 2칸이고 남는 칸은 0이다."""
    numbers = vocab if numbers is None else numbers
    ids = torch.zeros(len(texts), length + 2, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [START] + [numbers.get(ch, UNKNOWN) for ch in text][:length] + [END]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

X_tr, X_va, X_un = encode(texts_tr), encode(texts_va), encode(texts_un)
print(len(texts_tr), len(texts_va), len(texts_un), V + 2)
print(X_tr.shape, X_va.shape, X_un.shape)
shared = set(texts_un)
print(sum(text in shared for text in texts_va), round(y_va.float().mean().item(), 4))

In [ ]:
class SelfAttention(nn.Module):
    """12장의 셀프 어텐션. x: (배치, 길이, dim)."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)

    def forward(self, x, hide, causal):
        B, L, D = x.shape
        H = self.heads
        split = lambda t: t.view(B, L, H, D // H).transpose(1, 2)    # (B, H, L, D/H)
        q, k, v = split(self.to_q(x)), split(self.to_k(x)), split(self.to_v(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // H)         # (B, H, L, L)
        scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        if causal:                                                  # 뒤의 토큰을 가린다
            later = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(later, float("-inf"))
        mixed = torch.softmax(scores, dim=-1) @ v
        return self.out(mixed.transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    """층 정규화 → 셀프 어텐션 → 더하기, 층 정규화 → 완전 연결 층 둘 → 더하기."""
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = SelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(),
                                 nn.Linear(4 * dim, dim))

    def forward(self, x, hide, causal):
        x = x + self.attend(self.norm1(x), hide, causal)
        return x + self.mlp(self.norm2(x))

class Body(nn.Module):
    """몸통: 토큰 임베딩 + 위치 임베딩 → 블록 2개 → 층 정규화. 출력 층은 없다."""
    def __init__(self, dim=64, heads=4, layers=2, length=66):
        super().__init__()
        self.table = nn.Embedding(V + 2, dim)
        self.where = nn.Embedding(length, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids, causal):                          # (배치, 길이, 64)
        x = self.table(ids) + self.where(torch.arange(ids.shape[1]))
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide, causal)
        return self.norm(x)

class LanguageModel(nn.Module):
    """몸통 + 다음 토큰의 점수 V + 2개를 내는 출력 층."""
    def __init__(self):
        super().__init__()
        self.body, self.head = Body(), nn.Linear(64, V + 2)

    def forward(self, ids):                                  # (배치, 길이, V + 2)
        return self.head(self.body(ids, causal=True))

class Classifier(nn.Module):
    """몸통 + 긍정과 부정의 점수 2개를 내는 출력 층."""
    def __init__(self, causal):
        super().__init__()
        self.body, self.head, self.causal = Body(), nn.Linear(64, 2), causal

    def pooled(self, ids):                                   # 문장마다 벡터 하나
        x = self.body(ids, self.causal)
        real = (ids != PAD)[:, :, None].float()              # 패딩을 빼고 평균한다
        return (x * real).sum(dim=1) / real.sum(dim=1)

    def forward(self, ids):
        return self.head(self.pooled(ids))

count = lambda net: sum(p.numel() for p in net.parameters())
torch.manual_seed(0)
print(count(Body()), count(LanguageModel()), count(Classifier(causal=False)))
print(Classifier(causal=False)(X_tr[:8]).shape, LanguageModel()(X_tr[:8]).shape)

In [ ]:
def accuracy(net, X, y):
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(X[k:k + 1000]).argmax(dim=1)
                           for k in range(0, len(X), 1000)])
    return (guess == y).float().mean().item() * 100

def batches(lengths, seed, size=64, chunk=50):
    """길이가 비슷한 문장끼리 배치를 만든다. 섞은 뒤 3,200문장씩 길이 순으로 놓는다."""
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def fit(net, X, y, lr, seed=0, epochs=2, loss_of=None):
    """AdamW, 배치 크기 64. 학습률은 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    warm = max(1, total // 10)
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            wave = 0.5 * (1 + math.cos(math.pi * step / total))
            for group in optimizer.param_groups:
                group["lr"] = lr * min(1, step / warm) * wave
            ids = X[rows][:, :int(lengths[rows].max())]    # 배치에서 가장 긴 문장까지만
            optimizer.zero_grad()
            if loss_of is None:
                loss = F.cross_entropy(net(ids), y[rows])
            else:
                loss = loss_of(net, ids)
            loss.backward()
            optimizer.step()
    return net

SIZES = [300, 1000, 3000, 10000, 30000]                      # 쓸 레이블의 수
EPOCHS = {300: 20, 1000: 10, 3000: 4, 10000: 2, 30000: 2}   # 적으면 여러 번 본다
scores, took = {}, {}

def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def study(name, make_net, n, lr, X=None, X_test=None, seeds=5):
    """앞의 n문장으로 시드를 바꿔 학습시키고 검증 정확도(%)를 scores에 적는다."""
    X, X_test = (X_tr if X is None else X), (X_va if X_test is None else X_test)
    lap, marks = time.perf_counter(), []
    for seed in range(seeds):
        torch.manual_seed(seed)
        net = fit(make_net(), X[:n], y_tr[:n], lr, seed, EPOCHS[n])
        marks.append(accuracy(net, X_test, y_va))
    scores[name, n] = np.array(marks)
    took[name] = took.get(name, 0) + time.perf_counter() - lap

def show(*names, sizes=SIZES):
    print(pad("레이블의 수", 18) + "".join(f"{n:>12,}" for n in sizes))
    for name in names:
        marks = [scores[name, n] for n in sizes]
        print(pad(name, 18) + "".join(f"{np.median(s):>12.2f}" for s in marks))
        if len(marks[0]) > 1:
            spans = [f"{s.min():>6.2f}~{s.max():.2f}" for s in marks]
            print(pad("  범위", 18) + "".join(spans))

print([EPOCHS[n] * math.ceil(n / 64) for n in SIZES])        # 가중치를 고치는 걸음의 수
print([int(y_tr[:n].sum()) for n in SIZES])                  # 앞의 n문장 가운데 긍정

In [ ]:
def logistic(n):
    """11장의 기준선: 글자 1~3-그램의 TF-IDF와 로지스틱 회귀."""
    grams = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), min_df=2,
                            sublinear_tf=True)
    model = LogisticRegression(max_iter=1000, C=3)
    model.fit(grams.fit_transform(texts_tr[:n]), y_tr[:n])
    return model.score(grams.transform(texts_va), y_va) * 100, len(grams.vocabulary_)

kinds = {}
for n in SIZES:
    mark, kinds[n] = logistic(n)
    scores["로지스틱 회귀", n] = np.array([mark])
show("로지스틱 회귀")
print(pad("n-그램의 종류", 18) + "".join(f"{kinds[n]:>12,}" for n in SIZES))

In [ ]:
class RecNet(nn.Module):
    """11장의 LSTM 분류기. 마지막 토큰의 은닉 상태로 분류한다."""
    def __init__(self, dim=32, hidden=64):
        super().__init__()
        self.table = nn.Embedding(V + 2, dim, padding_idx=PAD)
        self.recurrent = nn.LSTM(dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 2)

    def forward(self, ids):
        states, _ = self.recurrent(self.table(ids))
        last = (ids != PAD).sum(dim=1) - 1                   # 마지막 진짜 토큰의 위치
        return self.head(states[torch.arange(len(ids)), last])

RATES = [0.0003, 0.001, 0.003, 0.01, 0.03]
sweep, best = {}, {}

def try_rates(name, make_net, rates=RATES, X=None, X_test=None, n=3000):
    """레이블 3,000개, 시드 0으로 학습률마다 검증 정확도를 재고 가장 좋은 값을 고른다"""
    X, X_test = (X_tr if X is None else X), (X_va if X_test is None else X_test)
    sweep[name] = []
    for lr in rates:
        torch.manual_seed(0)
        net = fit(make_net(), X[:n], y_tr[:n], lr, 0, EPOCHS[n])
        sweep[name].append(accuracy(net, X_test, y_va))
    best[name] = rates[int(np.argmax(sweep[name]))]
    print(pad(name, 18) + "".join(f"{a:>9.2f}" for a in sweep[name])
          + f"   고른 값 {best[name]}")

print(pad("학습률", 18) + "".join(f"{lr:>9}" for lr in RATES))
try_rates("트랜스포머", lambda: Classifier(causal=False))
try_rates("LSTM", lambda: RecNet())

In [ ]:
for n in SIZES:
    study("트랜스포머", lambda: Classifier(causal=False), n, best["트랜스포머"])
    study("LSTM", lambda: RecNet(), n, best["LSTM"])
show("로지스틱 회귀", "LSTM", "트랜스포머")
gap = [np.median(scores["트랜스포머", n]) - scores["로지스틱 회귀", n][0]
       for n in SIZES]
print(pad("트랜스포머 - 회귀", 18) + "".join(f"{g:>12.2f}" for g in gap))
print(count(RecNet()), count(Classifier(causal=False)))

## 13.2 레이블 없는 문장으로 언어 모델을 사전 학습시키기

In [ ]:
def lm_loss(net, ids):
    """앞 토큰들로 다음 토큰을 맞히는 교차 엔트로피. 패딩인 목표는 뺀다."""
    out = net(ids[:, :-1])                                   # 입력은 마지막 칸을 뺀 것
    goal = ids[:, 1:]                                        # 목표는 한 칸 뒤의 토큰
    return F.cross_entropy(out.reshape(-1, out.shape[-1]), goal.reshape(-1),
                           ignore_index=PAD)

def lm_score(net, data):
    """토큰 하나의 평균 손실."""
    net.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for k in range(0, len(data), 500):
            ids = data[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]),
                                     ids[:, 1:].reshape(-1), ignore_index=PAD,
                                     reduction="sum").item()
            n += (ids[:, 1:] != PAD).sum().item()
    return total / n

torch.manual_seed(0)
lap = time.perf_counter()
lm = fit(LanguageModel(), X_un, None, 0.03, epochs=2, loss_of=lm_loss)
took["사전 학습"] = time.perf_counter() - lap
lm_valid = lm_score(lm, X_va)
print(2 * math.ceil(len(X_un) / 64), int((X_un[:, 1:] != PAD).sum()))
print(round(math.log(V + 2), 4), round(lm_valid, 4), round(math.exp(lm_valid), 1))

torch.save(lm.body.state_dict(), "body.pt")                # 몸통의 가중치만 저장한다
pretrained = torch.load("body.pt")
print(type(pretrained).__name__, len(pretrained),
      sum(t.numel() for t in pretrained.values()))
for key in list(pretrained)[:4]:
    print(pad(key, 30), tuple(pretrained[key].shape))

In [ ]:
def pieces(texts):
    return [["<시작>"] + list(text)[:64] + ["<끝>"] for text in texts]

def three_gram(texts, mix=(0.1, 0.3, 0.6)):
    """texts에서 글자의 횟수를 세고, 12.4절의 3-그램 모델로 검증 손실을 구한다."""
    one, two, three = Counter(), defaultdict(Counter), defaultdict(Counter)
    for line in pieces(texts):
        for k in range(1, len(line)):
            before = line[k - 2] if k >= 2 else "<시작>"
            one[line[k]] += 1
            two[line[k - 1]][line[k]] += 1
            three[before, line[k - 1]][line[k]] += 1
    everything = sum(one.values())
    share = lambda found, token: found[token] / sum(found.values()) if found else 0
    total, n = 0.0, 0
    for line in pieces(texts_va):
        for k in range(1, len(line)):
            before = line[k - 2] if k >= 2 else "<시작>"
            p1 = (one[line[k]] + 1) / (everything + V + 2)
            p2 = share(two.get(line[k - 1]), line[k])
            p3 = share(three.get((before, line[k - 1])), line[k])
            total -= math.log(mix[0] * p1 + mix[1] * p2 + mix[2] * p3)
            n += 1
    return total / n

counted = {"30,000문장": three_gram(texts_tr), "109,995문장": three_gram(texts_un)}
print("다음 글자 예측의 검증 손실        3-그램    트랜스포머")
LM_CH12 = 3.3308                                          # 12.4절의 실행 결과
print(pad("학습 문장 30,000개 (12장)", 30)
      + f"{counted['30,000문장']:.4f}    {LM_CH12}")
print(pad("레이블 없는 문장 109,995개", 30)
      + f"{counted['109,995문장']:.4f}    {lm_valid:.4f}")

In [ ]:
names_of = {k: ch for ch, k in vocab.items()}

def generate(net, start, longest=30):
    """start 뒤에 확률이 가장 큰 토큰을 하나씩 붙인다(12.5절의 온도 0)."""
    ids = [START] + [vocab[ch] for ch in start]
    net.eval()
    for _ in range(longest):
        with torch.no_grad():
            out = net(torch.tensor([ids]))[0, -1]
        out[[PAD, UNKNOWN, START]] = float("-inf")
        if int(out.argmax()) == END:
            break
        ids.append(int(out.argmax()))
    return "".join(names_of[k] for k in ids[1:])

for start in ["이 영화", "내용이", "배우들", "마지막"]:
    print(generate(lm, start))

## 13.3 사전 학습한 층을 얼리고 출력 층만 학습시키기

In [ ]:
def classifier(state=None, causal=True, frozen=False):
    """분류기를 만든다. state가 있으면 몸통에 넣고, frozen이면 몸통을 얼린다."""
    net = Classifier(causal)
    if state is not None:
        net.body.load_state_dict(state)                      # 몸통의 가중치를 넣는다
    if frozen:
        for p in net.body.parameters():
            p.requires_grad_(False)                          # 기울기를 구하지 않는다
    return net

learnable = lambda net: sum(p.numel() for p in net.parameters() if p.requires_grad)
torch.manual_seed(0)
iced = classifier(pretrained, frozen=True)
print(count(iced), learnable(iced), count(iced.head))
print(torch.equal(iced.body.table.weight, lm.body.table.weight),
      torch.equal(iced.body.blocks[1].mlp[2].weight, lm.body.blocks[1].mlp[2].weight))

iced.train()
in_train = iced.pooled(X_tr[:8])                            # 학습 모드에서 계산한 벡터
iced.eval()
in_eval = iced.pooled(X_tr[:8])                             # 평가 모드에서 계산한 벡터
print(in_train.shape, in_train.requires_grad, float((in_train - in_eval).abs().max()))

In [ ]:
def pooled_all(net, X):
    """문장마다 몸통의 출력을 평균한 벡터 64개를 미리 계산해 둔다."""
    net.eval()
    made = []
    with torch.no_grad():
        for k in range(0, len(X), 1000):
            ids = X[k:k + 1000]
            made.append(net.pooled(ids[:, :int((ids != PAD).sum(dim=1).max())]))
    return torch.cat(made)

lap = time.perf_counter()
Z_tr, Z_va = pooled_all(iced, X_tr), pooled_all(iced, X_va)
torch.manual_seed(0)
blank = classifier(None, frozen=True)                       # 학습시키지 않은 몸통
R_tr, R_va = pooled_all(blank, X_tr), pooled_all(blank, X_va)
took["벡터 계산"] = time.perf_counter() - lap
print(Z_tr.shape, Z_va.shape, round(Z_tr.std().item(), 4), round(R_tr.std().item(), 4))

HEAD_RATES = [0.01, 0.03, 0.1, 0.3]
print(pad("학습률", 18) + "".join(f"{lr:>9}" for lr in HEAD_RATES))
try_rates("얼린 몸통", lambda: nn.Linear(64, 2), HEAD_RATES, Z_tr, Z_va)
try_rates("무작위 몸통", lambda: nn.Linear(64, 2), HEAD_RATES, R_tr, R_va)

In [ ]:
for n in SIZES:
    study("얼린 몸통", lambda: nn.Linear(64, 2), n, best["얼린 몸통"], Z_tr, Z_va)
    study("무작위 몸통", lambda: nn.Linear(64, 2), n, best["무작위 몸통"], R_tr, R_va)
show("트랜스포머", "얼린 몸통", "무작위 몸통")
gain = [np.median(scores["얼린 몸통", n]) - np.median(scores["무작위 몸통", n])
        for n in SIZES]
print(pad("얼린 - 무작위", 18) + "".join(f"{g:>12.2f}" for g in gain))

## 13.4 전체를 미세 조정하기: 학습률과 레이블의 수

In [ ]:
print(pad("학습률", 18) + "".join(f"{lr:>9}" for lr in RATES))
try_rates("미세 조정", lambda: classifier(pretrained))
print(pad("트랜스포머", 18) + "".join(f"{a:>9.2f}" for a in sweep["트랜스포머"]))
print(learnable(classifier(pretrained)))

In [ ]:
for n in SIZES:
    study("미세 조정", lambda: classifier(pretrained), n, best["미세 조정"])
show("로지스틱 회귀", "LSTM", "트랜스포머", "얼린 몸통", "미세 조정")
for other in ["트랜스포머", "로지스틱 회귀"]:
    gain = [np.median(scores["미세 조정", n]) - np.median(scores[other, n])
            for n in SIZES]
    print(pad("조정 - " + other[:5], 18) + "".join(f"{g:>12.2f}" for g in gain))
wins = [int((scores["미세 조정", n] > scores["트랜스포머", n].max()).sum())
        for n in SIZES]
print(wins)

## 13.5 사전 학습 조건 바꿔 보기와 공개된 사전 학습 모델 쓰기

In [ ]:
n = 10000
recount = Counter(ch for text in texts_tr[:n] for ch in text)
renumbered = {ch: k + 2 for k, (ch, c) in enumerate(recount.most_common()) if c >= 5}
same = sum(vocab.get(ch) == k for ch, k in renumbered.items())
print(len(vocab), len(renumbered), same)
word = "재밌다"
print([vocab[ch] for ch in word], [renumbered[ch] for ch in word],
      [names_of[renumbered[ch]] for ch in word])

W_tr, W_va = encode(texts_tr, renumbered), encode(texts_va, renumbered)
study("번호가 다른 어휘", lambda: classifier(pretrained), n, best["미세 조정"],
      W_tr, W_va)
study("마스크 없이 조정", lambda: classifier(pretrained, causal=False), n,
      best["미세 조정"])
study("새 어휘로 처음부터", lambda: Classifier(causal=False), n, best["트랜스포머"],
      W_tr, W_va)
show("트랜스포머", "새 어휘로 처음부터", "미세 조정", "번호가 다른 어휘",
     "마스크 없이 조정", sizes=[n])

In [ ]:
def continued(X_more):
    """사전 학습한 언어 모델을 X_more의 문장으로 3에폭 더 학습시킨다."""
    torch.manual_seed(0)
    net = LanguageModel()
    net.load_state_dict(lm.state_dict())
    return fit(net, X_more, None, 0.003, epochs=3, loss_of=lm_loss)

X_other = X_tr[20000:30000]                                 # 사전 학습에 없던 다른 문장
lap = time.perf_counter()
saw_other, saw_valid = continued(X_other), continued(X_va)
took["이어서 학습"] = time.perf_counter() - lap
leak = {}                                                   # (검증, 다른 문장)의 손실
print("토큰 하나의 손실                  검증 문장   다른 문장")
for key, name, net in [("base", "사전 학습한 모델", lm),
                       ("other", "다른 문장 10,000개를 더 봄", saw_other),
                       ("valid", "검증 문장 10,000개를 더 봄", saw_valid)]:
    leak[key] = (lm_score(net, X_va), lm_score(net, X_other))
    print(pad(name, 32) + f"{leak[key][0]:.4f}     {leak[key][1]:.4f}")

study("다른 문장을 본 몸통", lambda: classifier(saw_other.body.state_dict()), 1000,
      best["미세 조정"])
study("검증 문장을 본 몸통", lambda: classifier(saw_valid.body.state_dict()), 1000,
      best["미세 조정"])
show("미세 조정", "다른 문장을 본 몸통", "검증 문장을 본 몸통", sizes=[1000])

In [ ]:
amount = {0: math.log(V + 2), len(X_un): lm_valid}          # 문장의 수: 검증 손실
lap = time.perf_counter()
for m in [11000, 33000]:
    torch.manual_seed(0)
    small = fit(LanguageModel(), X_un[:m], None, 0.03, epochs=2, loss_of=lm_loss)
    amount[m] = lm_score(small, X_va)
    study(f"사전 학습 {m:,}문장", lambda: classifier(small.body.state_dict()), 10000,
          best["미세 조정"])
took["작은 사전 학습"] = time.perf_counter() - lap
print("사전 학습 문장의 수   언어 모델의 검증 손실   레이블 10,000개의 정확도 (범위)")
for m, name in [(0, "트랜스포머"), (11000, "사전 학습 11,000문장"),
                (33000, "사전 학습 33,000문장"), (len(X_un), "미세 조정")]:
    s = scores[name, 10000]
    print(f"{m:>12,}          {amount[m]:.4f}              {np.median(s):.2f} "
          f"({s.min():.2f}~{s.max():.2f})")

In [ ]:
RUN_PUBLIC = False                    # 코랩에서 GPU를 켜고 True로 바꾸면 실행된다
if RUN_PUBLIC:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    name = "klue/bert-base"                                  # 공개된 한국어 모델
    tokenizer = AutoTokenizer.from_pretrained(name)          # 그 모델의 어휘와 규칙
    public = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    public.to(device)

    def tokens(texts):
        made = tokenizer(texts, padding=True, truncation=True, max_length=64,
                         return_tensors="pt")
        return made.to(device)

    n = 1000                                                 # 쓸 레이블의 수
    optimizer = torch.optim.AdamW(public.parameters(), lr=2e-5)
    shuffler = torch.Generator().manual_seed(0)
    for epoch in range(3):
        public.train()
        for rows in torch.randperm(n, generator=shuffler).split(32):
            batch = tokens([texts_tr[k] for k in rows.tolist()])
            loss = public(**batch, labels=y_tr[rows].to(device)).loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    public.eval()
    right = 0
    with torch.no_grad():
        for k in range(0, len(texts_va), 200):
            guess = public(**tokens(texts_va[k:k + 200])).logits.argmax(dim=1).cpu()
            right += (guess == y_va[k:k + 200]).sum().item()
    print(count(public), round(right / len(texts_va) * 100, 2))

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
      sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())
for name, seconds in took.items():                           # 걸린 시간
    print(pad(name, 24) + f"{seconds:>5.0f}초")

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 13-1
from matplotlib.patches import Patch
fig, axes = plt.subplots(2, 1, figsize=(7, 9.3), gridspec_kw={"height_ratios": [1.6, 1]})
spots = np.arange(len(SIZES))
size_names = [f"{size:,}" for size in SIZES]
mid = lambda label: np.array([np.median(scores[label, size]) for size in SIZES])
ax = axes[0]
trio = [("로지스틱 회귀", "#6b7480", "s", "글자 1~3-그램 로지스틱 회귀 (기준선)"), ("LSTM", TEAL, "o", "LSTM (시드 5개의 중앙값)"),
        ("트랜스포머", NAVY, "D", "트랜스포머 (시드 5개의 중앙값)")]
for label, shade, mark, legend in trio:
    ax.plot(spots, mid(label), color=shade, marker=mark, lw=1.9, ms=6.5, label=legend, zorder=3)
    for spot, size in zip(spots, SIZES):
        if len(scores[label, size]) > 1:
            ax.scatter([spot] * 5, scores[label, size], s=9, color="white", edgecolor=shade, lw=0.6, zorder=4)
for spot in spots:
    ax.text(spot, mid("로지스틱 회귀")[spot] + 1.2, f"{mid('로지스틱 회귀')[spot]:.2f}", ha="center", fontsize=9.6, color="#4a525d", weight="bold")
    upper, lower = ("트랜스포머", "LSTM") if mid("트랜스포머")[spot] > mid("LSTM")[spot] else ("LSTM", "트랜스포머")
    for label, shift, anchor in [(upper, -0.1, "right"), (lower, 0.1, "left")]:
        tone = NAVY if label == "트랜스포머" else TEAL
        drop = 1.3 if label == upper else -1.5
        ax.text(spot + shift, mid(label)[spot] + drop, f"{mid(label)[spot]:.2f}", ha=anchor, va="center", fontsize=9.4, color=tone, weight="bold", bbox=BOX)
ax.axvspan(-0.4, 1.5, color=NAVY, alpha=0.05, lw=0); ax.axvspan(1.5, 4.75, color=TEAL, alpha=0.06, lw=0)
ax.text(0.55, 50.3, "레이블 1,000개까지:\n트랜스포머가 LSTM보다 높다", ha="center", fontsize=9, color=NAVY, linespacing=1.5)
ax.text(3.1, 50.3, "레이블 3,000개부터:\nLSTM이 트랜스포머보다 높다", ha="center", fontsize=9, color=TEAL, linespacing=1.5)
ax.set_xticks(spots); ax.set_xticklabels(size_names, fontsize=10); ax.set_xlim(-0.4, 4.75); ax.set_ylim(49, 90)
ax.set_xlabel("학습에 쓴 레이블의 수 (학습 문장 30,000개의 앞에서부터 자른 것)", fontsize=9); ax.set_ylabel("검증 문장 10,000개의 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9.2, loc="upper left")
frame(ax)
ax.set_title("① 레이블의 수를 줄였을 때의 검증 정확도 (흰 점은 시드 하나의 값)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
for shift, label, shade in [(-0.19, "LSTM", TEAL), (0.19, "트랜스포머", NAVY)]:
    behind = mid(label) - mid("로지스틱 회귀")
    ax.bar(spots + shift, behind, width=0.36, color=shade)
    for spot in spots:
        ax.text(spot + shift, behind[spot] - 0.35, f"{behind[spot]:+.2f}".replace("-", "$-$"), ha="center", va="top", fontsize=9.2, color=shade, weight="bold")
ax.axhline(0, color="#333", lw=0.9)
ax.text(4.7, 0.35, "0 = 기준선과 같다", fontsize=9, color="#333", ha="right")
ax.set_xticks(spots); ax.set_xticklabels(size_names, fontsize=10); ax.set_xlim(-0.4, 4.75); ax.set_ylim(-14.5, 1.6)
ax.set_xlabel("학습에 쓴 레이블의 수", fontsize=9); ax.set_ylabel("기준선과의 차이 (%p)", fontsize=9)
ax.legend(handles=[Patch(color=TEAL, label="LSTM $-$ 로지스틱 회귀"), Patch(color=NAVY, label="트랜스포머 $-$ 로지스틱 회귀")], frameon=False, fontsize=9.2,
          loc="lower right", ncol=2)
frame(ax)
ax.set_title("② 기준선과의 차이: 두 신경망은 레이블이 몇 개든 기준선보다 낮다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.06, hspace=0.26)
fig.savefig("fig_13_1.png", dpi=300)
plt.show()
print([round(float(v), 2) for v in mid("LSTM") - mid("로지스틱 회귀")], [round(float(v), 2) for v in mid("트랜스포머") - mid("LSTM")])

In [ ]:
# 그림 13-3
fig, axes = plt.subplots(2, 1, figsize=(7, 8.6), gridspec_kw={"height_ratios": [1.5, 1]})
ax = axes[0]
pairs = [("학습 문장 30,000개\n(12장)", counted["30,000문장"], LM_CH12), (f"레이블 없는 문장 {len(texts_un):,}개\n(이 장)", counted["109,995문장"], lm_valid)]
for spot, (label, by_count, by_net) in enumerate(pairs):
    for shift, value, shade in [(-0.19, by_count, GRAY), (0.19, by_net, NAVY)]:
        ax.bar(spot + shift, value, width=0.36, color=shade)
        ax.text(spot + shift, value + 0.012, f"{value:.4f}", ha="center", fontsize=10.5, color="#111", weight="bold")
        ax.text(spot + shift, 2.83, f"퍼플렉서티\n{math.exp(value):.1f}", ha="center", fontsize=9, color="white", weight="bold", linespacing=1.4)
    ax.text(spot - 0.19, by_count + 0.07, f"트랜스포머가\n{by_net - by_count:+.4f} 크다", ha="center", fontsize=9, color=BRICK, linespacing=1.45, weight="bold")
ax.annotate("", xy=(1.19, lm_valid + 0.03), xytext=(0.36, LM_CH12 - 0.01), arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=1.2, connectionstyle="arc3,rad=-0.18"))
ax.text(0.86, 3.345, f"문장을 늘리자 {lm_valid - LM_CH12:+.4f}".replace("-", "$-$"), fontsize=9.2, color=NAVY, ha="center", weight="bold", bbox=BOX)
ax.set_xticks(range(2)); ax.set_xticklabels([label for label, _, _ in pairs], fontsize=10, linespacing=1.4); ax.set_xlim(-0.55, 1.55)
ax.set_ylim(2.78, 3.52); ax.set_ylabel("검증 문장의 토큰 하나의 손실 (작을수록 좋다)", fontsize=9)
ax.legend(handles=[Patch(color=GRAY, label="글자를 세는 3-그램 모델"), Patch(color=NAVY, label="트랜스포머 언어 모델 (블록 2개, 2에폭)")], frameon=False, fontsize=9.2,
          loc="upper right")
frame(ax)
ax.set_title("① 다음 글자 예측의 검증 손실 (세로축은 2.78에서 시작)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
ax.axis("off")
starts = ["이 영화", "내용이", "배우들", "마지막"]
written = [generate(lm, start) for start in starts]
ax.text(0.0, 0.93, "사전 학습한 언어 모델이 이어 쓴 문장 (확률이 가장 큰 글자를 차례로 고른 것)", fontsize=9, color="#3a4350", va="center")
for row, (start, text) in enumerate(zip(starts, written)):
    height = 0.74 - row * 0.2
    ax.add_patch(plt.Rectangle((0.0, height - 0.075), 0.16, 0.15, fc="#eef1f5", ec=NAVY, lw=0.9, transform=ax.transAxes))
    ax.text(0.08, height, start, fontsize=10.5, color=NAVY, weight="bold", ha="center", va="center")
    ax.annotate("", xy=(0.215, height), xytext=(0.17, height), xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.1))
    ax.add_patch(plt.Rectangle((0.225, height - 0.075), 0.775, 0.15, fc="#f4faf8", ec=TEAL, lw=0.9, transform=ax.transAxes))
    ax.text(0.245, height, start, fontsize=10.5, color=NAVY, weight="bold", va="center")
    ax.text(0.257 + 0.0125 * sum(2 if ord(ch) > 0x1100 else 1 for ch in start), height, text[len(start):][:34], fontsize=10.5, color=TEAL, va="center")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("② 주어진 시작말(남색) 뒤에 모델이 붙인 글자(청록)", fontsize=10, loc="left", color=NAVY, weight="bold")
fig.subplots_adjust(left=0.1, right=0.97, top=0.96, bottom=0.03, hspace=0.2)
fig.savefig("fig_13_3.png", dpi=300)
plt.show()
print(written, [round(math.exp(v), 1) for _, a, b in pairs for v in (a, b)], round(lm_valid - counted["109,995문장"], 4), round(lm_valid - LM_CH12, 4))

In [ ]:
# 그림 13-5
fig, axes = plt.subplots(2, 1, figsize=(7, 9.3), gridspec_kw={"height_ratios": [1.6, 1]})
ax = axes[0]
quad = [("로지스틱 회귀", "#6b7480", "s", "로지스틱 회귀 (기준선)"), ("트랜스포머", NAVY, "D", "트랜스포머, 처음부터 학습"),
        ("얼린 몸통", BRICK, "o", "사전 학습한 몸통을 얼리고 출력 층만 학습"), ("무작위 몸통", GRAY, "^", "학습시키지 않은 몸통을 얼리고 출력 층만 학습")]
for label, shade, mark, legend in quad:
    ax.plot(spots, mid(label), color=shade, marker=mark, lw=1.9, ms=6.5, label=legend, zorder=3, ls="--" if label == "무작위 몸통" else "-")
for label, tone, lift in [("얼린 몸통", BRICK, 1.5), ("무작위 몸통", "#6b7480", -2.1), ("트랜스포머", NAVY, 1.5)]:
    for spot in spots:
        nudge = -2.2 if (label == "트랜스포머" and spot == 0) else lift
        if label == "얼린 몸통" and spot == 1: nudge = -2.1
        ax.text(spot, mid(label)[spot] + nudge, f"{mid(label)[spot]:.2f}", ha="center", va="center", fontsize=9.4, color=tone, weight="bold", bbox=BOX)
ax.annotate("", xy=(4.28, mid("얼린 몸통")[4]), xytext=(4.28, mid("무작위 몸통")[4]), arrowprops=dict(arrowstyle="<|-|>", color="#333", lw=1.0))
ax.text(4.34, (mid("얼린 몸통")[4] + mid("무작위 몸통")[4]) / 2, f"사전 학습의 몫\n{mid('얼린 몸통')[4] - mid('무작위 몸통')[4]:+.2f}%p", fontsize=9, color="#111", va="center", linespacing=1.45)
ax.set_xticks(spots); ax.set_xticklabels(size_names, fontsize=10); ax.set_xlim(-0.4, 5.15); ax.set_ylim(51, 92.5)
ax.set_xlabel("학습에 쓴 레이블의 수", fontsize=9); ax.set_ylabel("검증 문장 10,000개의 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left")
frame(ax)
ax.set_title("① 몸통을 얼렸을 때의 검증 정확도 (시드 5개의 중앙값)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
rise = [(legend, mid(label)[4] - mid(label)[0], shade, mid(label)[0], mid(label)[4])
        for label, shade, legend in [("로지스틱 회귀", "#6b7480", "로지스틱 회귀"), ("트랜스포머", NAVY, "트랜스포머, 처음부터"), ("얼린 몸통", BRICK, "얼린 몸통 + 출력 층"),
                                     ("무작위 몸통", GRAY, "무작위 몸통 + 출력 층")]]
for row, (legend, up, shade, low, high) in enumerate(rise):
    ax.barh(row, up, color=shade, height=0.6)
    ax.text(up + 0.3, row, f"+{up:.2f}%p", va="center", fontsize=10.5, color="#111", weight="bold")
    ax.text(up + 5.0, row, f"({low:.2f}% → {high:.2f}%)", va="center", fontsize=9, color="#4a525d")
ax.set_yticks(range(4)); ax.set_yticklabels([legend for legend, *_ in rise], fontsize=9.6); ax.invert_yaxis(); ax.set_xlim(0, 28.5)
ax.set_xlabel("레이블을 300개에서 30,000개로 늘렸을 때 오른 정확도 (%p)", fontsize=9)
frame(ax, grid="x")
ax.set_title("② 레이블을 100배로 늘렸을 때 오른 정확도: 얼린 몸통은 거의 오르지 않는다", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.3)
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.235, right=0.97, top=0.965, bottom=0.06, hspace=0.25)
fig.savefig("fig_13_5.png", dpi=300)
plt.show()
print([(legend, round(float(up), 2)) for legend, up, *_ in rise])

In [ ]:
# 그림 13-6
fig, ax = plt.subplots(figsize=(7, 6.4))
for label, shade, mark, legend in [("미세 조정", BRICK, "o", "사전 학습한 몸통에서 시작 (미세 조정)"), ("트랜스포머", NAVY, "D", "무작위 초기화에서 시작 (처음부터 학습)")]:
    ax.plot(RATES, sweep[label], color=shade, marker=mark, lw=1.9, ms=7, label=legend, zorder=3)
    for lr, value in zip(RATES, sweep[label]):
        lift = 1.5 if label == "미세 조정" else -1.6
        ax.text(lr, value + lift, f"{value:.2f}", ha="center", va="center", fontsize=9.8, color=shade, weight="bold", bbox=BOX)
top = max(sweep["트랜스포머"])
ax.axhline(top, color=NAVY, lw=0.9, ls=":")
ax.text(0.00027, top + 0.5, f"처음부터 학습의 가장 좋은 값 {top:.2f}%", fontsize=9, color=NAVY)
weak = sweep["미세 조정"][1]
ax.annotate(f"학습률 0.001로 미세 조정하면 {weak:.2f}%.\n처음부터 학습(0.01)의 {top:.2f}%보다 낮다", xy=(0.001, weak), xytext=(0.00026, 76.3), fontsize=9, color="#111", linespacing=1.5,
            arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.0))
ax.annotate("학습률 0.03: 처음부터 학습은\n무너지고 미세 조정은 버틴다", xy=(0.03, sweep["트랜스포머"][4]), xytext=(0.0019, 54.6), fontsize=9, color="#111", linespacing=1.5,
            arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.0))
ax.axvline(best["미세 조정"], color="#888", lw=0.8, ls="--", zorder=1)
ax.text(best["미세 조정"] * 1.06, 50.6, "두 방법 모두\n0.01을 골랐다", fontsize=9, color="#555", linespacing=1.4)
ax.set_xscale("log"); ax.set_xticks(RATES); plain_ticks(ax.xaxis); ax.set_xlim(0.00022, 0.042); ax.set_ylim(49.5, 85.5)
ax.set_xlabel("학습률 (로그 축)", fontsize=9); ax.set_ylabel("검증 문장 10,000개의 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="upper left")
frame(ax)
ax.set_title("레이블 3,000개, 시드 0에서 학습률에 따른 검증 정확도", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.94, bottom=0.09)
fig.savefig("fig_13_6.png", dpi=300)
plt.show()
print([round(float(a - b), 2) for a, b in zip(sweep["미세 조정"], sweep["트랜스포머"])])

In [ ]:
# 그림 13-7
fig, axes = plt.subplots(2, 1, figsize=(7, 9.4), gridspec_kw={"height_ratios": [1.65, 1]})
ax = axes[0]
five = [("로지스틱 회귀", "#6b7480", "s", "로지스틱 회귀 (기준선)", "-"), ("미세 조정", BRICK, "o", "사전 학습 + 전체 미세 조정", "-"),
        ("LSTM", TEAL, "v", "LSTM, 처음부터 학습", "-"), ("트랜스포머", NAVY, "D", "트랜스포머, 처음부터 학습", "-"), ("얼린 몸통", BRICK, "x", "사전 학습 + 얼린 몸통", ":")]
for label, shade, mark, legend, style in five:
    ax.plot(spots, mid(label), color=shade, marker=mark, lw=2.3 if label == "미세 조정" else 1.5, ms=6.5, label=legend, zorder=4 if label == "미세 조정" else 3, ls=style)
for spot in spots:
    ax.text(spot - 0.07, mid("미세 조정")[spot] + 1.5, f"{mid('미세 조정')[spot]:.2f}", ha="right", va="center", fontsize=9.8, color=BRICK, weight="bold", bbox=BOX)
    ax.text(spot, mid("로지스틱 회귀")[spot] + 1.5, f"{mid('로지스틱 회귀')[spot]:.2f}", ha="center", va="center", fontsize=9.2, color="#4a525d", weight="bold")
for label, shade, _, _, _ in five[2:]:
    ax.text(4.13, mid(label)[4], f"{mid(label)[4]:.2f}", va="center", fontsize=9.4, color=shade, weight="bold")
ax.scatter(np.repeat(spots, 5), np.concatenate([scores["미세 조정", size] for size in SIZES]), s=9, color="white", edgecolor=BRICK, lw=0.6, zorder=5)
ax.set_xticks(spots); ax.set_xticklabels(size_names, fontsize=10); ax.set_xlim(-0.45, 4.7); ax.set_ylim(54, 92.5)
ax.set_xlabel("학습에 쓴 레이블의 수", fontsize=9); ax.set_ylabel("검증 문장 10,000개의 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left", ncol=2, columnspacing=1.2)
frame(ax)
ax.set_title("① 다섯 가지 방법의 검증 정확도 (시드 5개의 중앙값)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
lift_scratch, lift_base = mid("미세 조정") - mid("트랜스포머"), mid("미세 조정") - mid("로지스틱 회귀")
ax.bar(spots - 0.19, lift_scratch, width=0.36, color=BRICK); ax.bar(spots + 0.19, lift_base, width=0.36, color=GRAY)
for spot in spots:
    ax.text(spot - 0.19, lift_scratch[spot] + 0.25, f"+{lift_scratch[spot]:.2f}", ha="center", fontsize=9.8, color=BRICK, weight="bold")
    ax.text(spot + 0.19, lift_base[spot] - 0.3, f"{lift_base[spot]:.2f}".replace("-", "$-$"), ha="center", va="top", fontsize=9.8, color="#4a525d", weight="bold")
ax.axhline(0, color="#333", lw=0.9)
ax.set_xticks(spots); ax.set_xticklabels(size_names, fontsize=10); ax.set_xlim(-0.45, 4.7); ax.set_ylim(-10.4, 6.6)
ax.set_xlabel("학습에 쓴 레이블의 수", fontsize=9); ax.set_ylabel("정확도의 차이 (%p)", fontsize=9)
ax.legend(handles=[Patch(color=BRICK, label="미세 조정 $-$ 처음부터 학습한 트랜스포머"), Patch(color=GRAY, label="미세 조정 $-$ 로지스틱 회귀")], frameon=False, fontsize=9.2,
          loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
frame(ax)
ax.set_title("② 사전 학습으로 오른 정확도(벽돌색)와 기준선과의 차이(회색)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.06, hspace=0.25)
fig.savefig("fig_13_7.png", dpi=300)
plt.show()
print([round(float(v), 2) for v in lift_scratch], [round(float(v), 2) for v in lift_base], round(float(mid("미세 조정")[4] - mid("LSTM")[4]), 2))

In [ ]:
# 그림 13-9
fig, axes = plt.subplots(2, 1, figsize=(7, 9.3), gridspec_kw={"height_ratios": [1.25, 1]})
ax = axes[0]
rows9 = [("미세 조정", "사전 학습 때와 같은 입력", BRICK), ("번호가 다른 어휘", "글자 번호를 새로 매김", "#d39a7e"), ("마스크 없이 조정", "인과 마스크를 뺌", "#d39a7e"),
         ("트랜스포머", "사전 학습 없음 (처음부터)", NAVY), ("새 어휘로 처음부터", "사전 학습 없음, 새 번호", "#7f93a8")]
anchor = np.median(scores["미세 조정", 10000])
for row, (label, legend, shade) in enumerate(rows9):
    marks = scores[label, 10000]
    ax.barh(row, np.median(marks), color=shade, height=0.62)
    ax.scatter(marks, [row] * 5, s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(79.0, row, f"{np.median(marks):.2f}%", va="center", fontsize=10.5, color="#111", weight="bold")
    if row in (1, 2):
        ax.text(80.75, row, f"{np.median(marks) - anchor:+.2f}%p".replace("-", "$-$"), va="center", fontsize=9.6, color=BRICK, weight="bold")
ax.text(80.75, 0, "기준", va="center", fontsize=9.6, color=BRICK)
ax.axvline(anchor, color=BRICK, lw=0.9, ls="--")
ax.set_yticks(range(5)); ax.set_yticklabels([legend for _, legend, _ in rows9], fontsize=9.8); ax.invert_yaxis(); ax.set_xlim(69, 82.6)
ax.set_xlabel("레이블 10,000개로 학습한 검증 정확도 (%). 막대는 시드 5개의 중앙값, 흰 점은 시드 하나", fontsize=9, x=0.42)
frame(ax, grid="x")
ax.set_title("① 사전 학습 때와 다르게 입력을 만들었을 때 (가로축은 69%에서 시작)", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.36)
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
ax = axes[1]
seen9 = [("사전 학습한\n모델", "base", GRAY), ("다른 문장 10,000개를\n더 본 모델", "other", NAVY), ("검증 문장 10,000개를\n더 본 모델", "valid", BRICK)]
for spot, (legend, key, shade) in enumerate(seen9):
    on_valid, on_other = leak[key]
    ax.bar(spot - 0.19, on_valid, width=0.36, color=shade); ax.bar(spot + 0.19, on_other, width=0.36, color=shade, alpha=0.45)
    ax.text(spot - 0.19, on_valid + 0.006, f"{on_valid:.4f}", ha="center", fontsize=10, color="#111", weight="bold")
    ax.text(spot + 0.19, on_other + 0.006, f"{on_other:.4f}", ha="center", fontsize=10, color="#555", weight="bold")
    ax.text(spot - 0.19, 2.865, "검증\n문장", ha="center", fontsize=9, color="white", weight="bold", linespacing=1.3)
    ax.text(spot + 0.19, 2.865, "다른\n문장", ha="center", fontsize=9, color="#222", weight="bold", linespacing=1.3)
honest, peeked = leak["other"][0], leak["valid"][0]
ax.annotate("", xy=(1.81, peeked + 0.022), xytext=(0.81, honest + 0.022), arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.2, connectionstyle="arc3,rad=-0.25"))
ax.text(1.3, 3.079, f"검증 문장을 보면 검증 손실이 {peeked - honest:+.4f}".replace("-", "$-$"), ha="center", fontsize=9.4, color=BRICK, weight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels([legend for legend, _, _ in seen9], fontsize=9.6, linespacing=1.4); ax.set_xlim(-0.55, 2.55)
ax.set_ylim(2.84, 3.1); ax.set_ylabel("토큰 하나의 손실", fontsize=9)
frame(ax)
ax.set_title("② 언어 모델을 3에폭 더 학습시킨 뒤의 손실 (진한 막대: 검증 문장, 옅은 막대: 다른 문장)", fontsize=10, loc="left", color=NAVY, weight="bold",
             x=-0.09)
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.27, right=0.97, top=0.965, bottom=0.075, hspace=0.3)
pos = axes[1].get_position(); axes[1].set_position([0.1, pos.y0, 0.87, pos.height])
fig.savefig("fig_13_9.png", dpi=300)
plt.show()
print([round(float(np.median(scores[label, 10000]) - anchor), 2) for label, _, _ in rows9], round(peeked - honest, 4),
      [round(float(np.median(scores[label, 1000])), 2) for label in ["미세 조정", "다른 문장을 본 몸통", "검증 문장을 본 몸통"]])

In [ ]:
# 그림 13-10
fig, axes = plt.subplots(3, 1, figsize=(7, 9.5), gridspec_kw={"height_ratios": [1, 1.25, 0.8]})
steps10 = [(0, "트랜스포머"), (11000, "사전 학습 11,000문장"), (33000, "사전 학습 33,000문장"), (len(texts_un), "미세 조정")]
names10 = ["0\n(처음부터 학습)", "11,000", "33,000", f"{len(texts_un):,}"]
ax = axes[0]
shown = [amount[m] for m, _ in steps10]
ax.bar(range(4), shown, color=[GRAY, NAVY, NAVY, NAVY], width=0.56)
for spot, value in enumerate(shown):
    ax.text(spot, value + 0.12, f"{value:.4f}", ha="center", fontsize=10.5, color="#111", weight="bold")
ax.text(0, 3.4, "학습 전:\n모든 토큰에\n같은 확률", ha="center", fontsize=9, color="white", weight="bold", linespacing=1.4)
ax.axhline(counted["109,995문장"], color=BRICK, lw=0.9, ls="--")
ax.text(3.45, 7.7, f"점선: 문장 {len(texts_un):,}개로 센 3-그램 {counted['109,995문장']:.4f}", fontsize=9, color=BRICK, ha="right")
ax.set_xticks(range(4)); ax.set_xticklabels(names10, fontsize=9.6); ax.set_xlim(-0.5, 3.5); ax.set_ylim(0, 8.6)
ax.set_ylabel("언어 모델의 검증 손실", fontsize=9)
frame(ax)
ax.set_title("① 사전 학습 문장의 수와 언어 모델의 검증 손실 (2에폭, 시드 0)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
floor10 = np.median(scores["트랜스포머", 10000])
for spot, (m, label) in enumerate(steps10):
    marks = scores[label, 10000]
    middle = np.median(marks)
    shade = GRAY if spot == 0 else (BRICK if middle < floor10 else TEAL)
    ax.bar(spot, middle, color=shade, width=0.56)
    ax.scatter([spot] * 5, marks, s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(spot, max(marks.max(), middle) + 0.45, f"{middle:.2f}%", ha="center", fontsize=10.5, color="#111", weight="bold")
    if spot:
        ax.text(spot, 63.1, f"{middle - floor10:+.2f}%p".replace("-", "$-$"), ha="center", fontsize=10, color="white", weight="bold")
ax.axhline(floor10, color="#333", lw=0.9, ls="--")
ax.annotate("문장 11,000개로 사전 학습한 몸통은\n처음부터 학습한 것보다 낮다", xy=(1.0, 67.6), xytext=(0.34, 77.4), fontsize=9, color=BRICK, linespacing=1.5,
            arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.0))
ax.set_xticks(range(4)); ax.set_xticklabels(names10, fontsize=9.6); ax.set_xlim(-0.5, 3.5); ax.set_ylim(62, 81.5)
ax.set_ylabel("검증 정확도 (%)", fontsize=9)
frame(ax)
ax.set_title("② 그 몸통을 레이블 10,000개로 미세 조정한 검증 정확도 (세로축은 62%에서 시작)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[2]
body_size = count(Body())
megabytes = sum(len(text.encode("utf-8")) for text in texts_un) / 1e6
scale10 = [("사전 학습에 쓴 글 (MB)", megabytes, 62000, f"{megabytes:.1f}MB", "약 62GB"),
           ("파라미터의 수", body_size, 110e6, f"{body_size:,}개", "약 1억 1천만 개")]
for row, (legend, ours, theirs, ours_text, theirs_text) in enumerate(scale10):
    ax.barh(row - 0.19, ours, height=0.34, color=NAVY); ax.barh(row + 0.19, theirs, height=0.34, color=TEAL)
    ax.text(ours * 1.25, row - 0.19, f"이 장의 모델: {ours_text}", va="center", fontsize=9.4, color=NAVY, weight="bold")
    ax.text(theirs * 1.25, row + 0.19, f"KLUE-BERT base: {theirs_text} ({theirs / ours:,.0f}배)", va="center", fontsize=9.4, color=TEAL, weight="bold")
ax.set_xscale("log"); ax.set_xlim(1, 2e15); ax.set_yticks(range(2)); ax.set_yticklabels([legend for legend, *_ in scale10], fontsize=9.6); ax.invert_yaxis()
ax.set_xticks([]); ax.spines[["top", "right", "bottom"]].set_visible(False); ax.tick_params(axis="y", length=0)
ax.set_title("③ 공개된 사전 학습 모델과의 크기 비교 (로그 축. KLUE-BERT의 값은 참고 문헌 3에서 옮긴 것)", fontsize=10, loc="left", color=NAVY, weight="bold",
             x=-0.27)
fig.subplots_adjust(left=0.215, right=0.97, top=0.965, bottom=0.03, hspace=0.42)
for axis in axes[:2]:
    pos = axis.get_position(); axis.set_position([0.1, pos.y0, 0.87, pos.height])
fig.savefig("fig_13_10.png", dpi=300)
plt.show()
print(round(megabytes, 1), body_size, round(62000 / megabytes), round(110e6 / body_size))